In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDac-AGL/AgentDaC/notebooks


In [3]:
import shlex

vllm_serve = [
    "vllm",
    "serve",
    MODEL_NAME,
    "--host",
    HOST,
    "--port",
    str(PORT),
    "--dtype",
    "bfloat16",
    "--max_model_len",
    "14336",
    "--max_num_seqs",
    "1024",
    "--enable_chunked_prefill",
    "--max_num_batched_tokens",
    "8192",
    "--enable_prefix_caching",
    "--logprobs_mode",
    "processed_logprobs",
    "--gpu_memory_utilization",
    "0.9",
    "--disable_log_stats",
    "--tensor_parallel_size",
    "1",
    "--seed",
    "0",
    "--generation_config",
    "auto",
    "--additional_config",
    '{"gdn_prefill_backend": "triton"}',
    "--reasoning_config",
    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}',
    "--reasoning-parser",
    "qwen3",
    "--tool-call-parser",
    "qwen3_xml",
    "--enable-auto-tool-choice",
]


def print_vllm_serve_command():
    print("vllm serve command:")
    print(shlex.join(vllm_serve))


print_vllm_serve_command()

vllm serve command:
vllm serve Qwen/Qwen3.5-4B --host 127.0.0.1 --port 43667 --dtype bfloat16 --max_model_len 14336 --max_num_seqs 1024 --enable_chunked_prefill --max_num_batched_tokens 8192 --enable_prefix_caching --logprobs_mode processed_logprobs --gpu_memory_utilization 0.9 --disable_log_stats --tensor_parallel_size 1 --seed 0 --generation_config auto --additional_config '{"gdn_prefill_backend": "triton"}' --reasoning_config '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}' --reasoning-parser qwen3 --tool-call-parser qwen3_xml --enable-auto-tool-choice


In [4]:
from src.inference import OAIClient

# HOST = "127.0.0.1"
PORT = "55307"

base_url = f"http://{HOST}:{PORT}/v1"

client = OAIClient(model_name=MODEL_NAME, base_url=base_url)

In [5]:
from experiments.chess.format import format_prompt

fens = [
    "r4rk1/5ppp/pQ1b1q2/8/1P6/P4N1P/5PP1/3R1RK1 b - - 2 25",
    "r5k1/2n5/5r1p/p1pP1N2/PpP2p2/1P1Q4/3B1q1N/7K w - - 0 33",
    "R4bk1/5rP1/5pR1/8/2r5/p5P1/6K1/1q6 w - - 0 42",
    "2r4k/p5pp/1p2q3/2p3Q1/8/7P/PB3PP1/6K1 w - - 0 30",
    "2Q5/1p3kp1/3p2p1/3Pp3/2N1n3/P7/KQq5/8 b - - 1 33",
    "1rr5/pbRp3p/1p1nkpp1/4p3/1B2P1P1/3B1P1P/PP2K3/2R5 w - - 8 25",
    "Q7/4ppbk/6pp/5q2/P3n3/4PNBP/1r3PP1/3R2K1 b - - 0 31",
    "1r3rk1/pbp3bp/6p1/2qN1p2/1P2B3/5Q2/P1P2PPP/1R3RK1 b - - 0 16",
]


samples = [{"fen": fen} for fen in fens]
prompts = [format_prompt(sample) for sample in samples]

In [ ]:
from src.agents import PersistentAgent, MarkerAgent, ToolSubmitAgent
from src.agents.tool_agent import build_tool_parser
from src.configs import PromptConfig, DecompConfig


kwargs = {
    "temperature": 1.0,
    "top_p": 0.95,
    # disable automatic tool parser selection from vLLM backend. 
    # Needed for front-end tool call parsing. This doesnt disable generating tool-calls
    # "tool_choice": "none",  
    "extra_body": {
        # "chat_template_kwargs": {"enable_thinking": False},
        "min_tokens": 5,
        "top_k": 20,
        "repretition_penalty": 1.0,
        "thinking_token_budget": 1024,
        "max_new_tokens": 12288,
    },
}

prompt_config = PromptConfig(
    mode="path",
    system_root="../config_files/prompts/tool/v1_root.txt",
    system_inter="../config_files/prompts/tool/v1_root.txt",
    system_leaf="../config_files/prompts/tool/v1_leaf.txt",
)

decomp_config = DecompConfig(
    max_depth=1,
    max_tasks=3,
    max_rounds=5,
)

tool_parser = build_tool_parser(
    tool_parser="qwen3_xml",
    tokenizer=MODEL_NAME,
    reasoning_parser="qwen3",
)

responses = []

for i, prompt in enumerate(prompts[:2]):
    print("--------------------" * 10)

    message = {"role": "user", "content": prompt}

    agent = ToolSubmitAgent(
        client=client,
        prompt_config=prompt_config,
        decomp_config=decomp_config,
        tool_parser=tool_parser,
        additional_histories=True,
        verbose=True,
    )

    response = await agent.chat(prompt=message, **kwargs)
    responses.append(response)

INFO 07-20 13:55:28 [qwen3xml_tool_parser.py:1178] vLLM Successfully import tool parser Qwen3XMLToolParser !
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Role:
SYSTEM
Content:
You are a helpful assistant that excels at solving tasks. 
You solve a task by reasoning and, when it helps, breaking it into smaller sub-tasks.

Keep your internal reasoning brief and to the point.

Each round, do exactly one of:
- Delegate a focused sub-task using one of the delegation tools available to you (e.g. `create_new_sub_agent`); you will receive the sub-agent's answer back and can continue.
- Or, when you have what you need, write your final answer directly as plain text, in the exact format the task requests. Making no tool call finishes the task.

A sub-agent is a blank slate: it has NO memory and NO knowledge of this conversation or the original

In [7]:
for i, response in enumerate(responses):
    print("--------------------" * 10)
    print(response.for_logging())

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
{'reward': 0.0, 'metrics': {'direct_calls': 2, 'subtree_calls': 3, 'direct_tasks': 1, 'subtree_tasks': 1, 'direct_thinks': 2, 'subtree_thinks': 3, 'direct_agents': 1, 'subtree_agents': 1, 'direct_chats': 1, 'subtree_chats': 2, 'subtree_depth': 1, 'direct_tokens': 2227, 'duration': 13.04791}, 'metadata': {}, 'messages': [{'role': 'system', 'content': "You are a helpful assistant that excels at solving tasks. \nYou solve a task by reasoning and, when it helps, breaking it into smaller sub-tasks.\n\nKeep your internal reasoning brief and to the point.\n\nEach round, do exactly one of:\n- Delegate a focused sub-task using one of the delegation tools available to you (e.g. `create_new_sub_agent`); you will receive the sub-agent's answer back and can continue.\n- Or, when you have what you need